# Improving the EN→RU Seq2Seq Translator
Extends the base Seq2Seq model with:
1. **BLEU evaluation** – implemented from scratch (no libraries)
2. **Beam search decoding** – compared against greedy decoding

## 0. Imports & Setup

In [1]:
import re
import math
import time
import random
import heapq
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## 1. Rebuild Base Model
Re-run the full pipeline from the base notebook so this notebook is self-contained.

In [2]:
# ── Special tokens ──
PAD_TOKEN = '<pad>'
UNK_TOKEN = '<unk>'
SOS_TOKEN = '<sos>'
EOS_TOKEN = '<eos>'
SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN]


def clean_text(text):
    """Lowercase, remove punctuation and digits, collapse whitespace."""
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def load_data(filepath, max_samples=None):
    """Load tab-separated EN-RU pairs (3-column format; third column ignored)."""
    en_sentences, ru_sentences = [], []
    with open(filepath, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if max_samples and i >= max_samples:
                break
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                en = clean_text(parts[0])
                ru = clean_text(parts[1])
                if en and ru:
                    en_sentences.append(en)
                    ru_sentences.append(ru)
    print(f'Loaded {len(en_sentences)} sentence pairs')
    return en_sentences, ru_sentences


class Vocabulary:
    def __init__(self, name, min_freq=2):
        self.name = name
        self.min_freq = min_freq
        self.word2idx = {t: i for i, t in enumerate(SPECIAL_TOKENS)}
        self.idx2word = {i: t for t, i in self.word2idx.items()}

    def build(self, sentences):
        counter = Counter()
        for sent in sentences:
            counter.update(sent.split())
        for word, freq in counter.items():
            if freq >= self.min_freq and word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word
        print(f'{self.name} vocabulary size: {len(self.word2idx)}')

    def encode(self, sentence):
        unk_idx = self.word2idx[UNK_TOKEN]
        return [self.word2idx.get(w, unk_idx) for w in sentence.split()]

    def decode(self, indices):
        return [self.idx2word.get(i, UNK_TOKEN) for i in indices]

    def __len__(self):
        return len(self.word2idx)


# ── Load & split ──
from sklearn.model_selection import train_test_split

DATA_PATH = 'rus.txt'
en_sentences, ru_sentences = load_data(DATA_PATH)

en_vocab = Vocabulary('English', min_freq=2)
ru_vocab = Vocabulary('Russian', min_freq=2)
en_vocab.build(en_sentences)
ru_vocab.build(ru_sentences)

pairs = list(zip(en_sentences, ru_sentences))
train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=SEED)
print(f'Train: {len(train_pairs)} | Val: {len(val_pairs)}')

Loaded 363386 sentence pairs
English vocabulary size: 11609
Russian vocabulary size: 31716
Train: 327047 | Val: 36339


In [3]:
# ── Dataset & DataLoader ──

PAD_IDX = en_vocab.word2idx[PAD_TOKEN]


class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab):
        self.pairs = pairs
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_sent, tgt_sent = self.pairs[idx]
        src_indices = self.src_vocab.encode(src_sent)
        sos = self.tgt_vocab.word2idx[SOS_TOKEN]
        eos = self.tgt_vocab.word2idx[EOS_TOKEN]
        tgt_indices = [sos] + self.tgt_vocab.encode(tgt_sent) + [eos]
        return (torch.tensor(src_indices, dtype=torch.long),
                torch.tensor(tgt_indices, dtype=torch.long))


def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_padded = pad_sequence(src_batch, batch_first=False, padding_value=PAD_IDX)
    tgt_padded = pad_sequence(tgt_batch, batch_first=False, padding_value=PAD_IDX)
    return src_padded, tgt_padded


BATCH_SIZE = 128
train_dataset = TranslationDataset(train_pairs, en_vocab, ru_vocab)
val_dataset   = TranslationDataset(val_pairs,   en_vocab, ru_vocab)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

Train batches: 2556 | Val batches: 284


In [4]:
# ── Model Architecture ──

class Encoder(nn.Module):
    """LSTM Encoder: maps source sentence to (hidden, cell) context."""
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=2, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                                 dropout=dropout if num_layers > 1 else 0,
                                 batch_first=False)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        _, (hidden, cell) = self.lstm(embedded)
        return hidden, cell


class Decoder(nn.Module):
    """LSTM Decoder: generates one target token per step."""
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=2, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                                 dropout=dropout if num_layers > 1 else 0,
                                 batch_first=False)
        self.fc_out    = nn.Linear(hidden_dim, vocab_size)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, tgt_token, hidden, cell):
        tgt_token = tgt_token.unsqueeze(0)                       # (1, batch)
        embedded  = self.dropout(self.embedding(tgt_token))      # (1, batch, embed_dim)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(0))              # (batch, vocab_size)
        return prediction, hidden, cell


class Seq2Seq(nn.Module):
    """Full encoder-decoder with configurable teacher-forcing ratio."""
    def __init__(self, encoder, decoder, tgt_vocab_size, sos_idx, eos_idx, device):
        super().__init__()
        self.encoder        = encoder
        self.decoder        = decoder
        self.tgt_vocab_size = tgt_vocab_size
        self.sos_idx        = sos_idx
        self.eos_idx        = eos_idx
        self.device         = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        tgt_len, batch_size = tgt.shape
        outputs = torch.zeros(tgt_len - 1, batch_size,
                              self.tgt_vocab_size, device=self.device)
        hidden, cell = self.encoder(src)
        dec_input = tgt[0]   # <sos>
        for t in range(1, tgt_len):
            pred, hidden, cell = self.decoder(dec_input, hidden, cell)
            outputs[t - 1] = pred
            teacher_force = random.random() < teacher_forcing_ratio
            top_idx = pred.argmax(1)
            dec_input = tgt[t] if teacher_force else top_idx
        return outputs


# ── Instantiate model ──
EMBED_DIM  = 256
HIDDEN_DIM = 512
NUM_LAYERS = 2
DROPOUT    = 0.5

SOS_IDX = ru_vocab.word2idx[SOS_TOKEN]
EOS_IDX = ru_vocab.word2idx[EOS_TOKEN]

encoder = Encoder(len(en_vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(device)
decoder = Decoder(len(ru_vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(device)
model   = Seq2Seq(encoder, decoder, len(ru_vocab), SOS_IDX, EOS_IDX, device).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,}')

Model parameters: 34,717,924


In [5]:
# ── Training loop ──

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
CLIP      = 1.0


def train_epoch(model, loader, optimizer, criterion, clip, teacher_forcing_ratio=0.5):
    model.train()
    epoch_loss = 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()
        output = model(src, tgt, teacher_forcing_ratio)
        output_flat = output.view(-1, output.shape[-1])
        tgt_flat    = tgt[1:].contiguous().view(-1)
        loss = criterion(output_flat, tgt_flat)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)


def evaluate_epoch(model, loader, criterion):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            output = model(src, tgt, teacher_forcing_ratio=0.0)
            output_flat = output.view(-1, output.shape[-1])
            tgt_flat    = tgt[1:].contiguous().view(-1)
            loss = criterion(output_flat, tgt_flat)
            epoch_loss += loss.item()
    return epoch_loss / len(loader)


import time

N_EPOCHS      = 10
best_val_loss = float('inf')
train_losses, val_losses = [], []

for epoch in range(1, N_EPOCHS + 1):
    start = time.time()
    train_loss = train_epoch(model, train_loader, optimizer, criterion, CLIP)
    val_loss   = evaluate_epoch(model, val_loader, criterion)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    elapsed = time.time() - start
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pt')
        marker = '  ✓ saved'
    else:
        marker = ''
    print(f'Epoch {epoch:02d}/{N_EPOCHS} | Train Loss: {train_loss:.4f} | '
          f'Val Loss: {val_loss:.4f} | Time: {elapsed:.1f}s{marker}')

Epoch 01/10 | Train Loss: 4.3721 | Val Loss: 3.4225 | Time: 667.8s  ✓ saved
Epoch 02/10 | Train Loss: 2.8895 | Val Loss: 2.8089 | Time: 958.6s  ✓ saved
Epoch 03/10 | Train Loss: 2.3597 | Val Loss: 2.5455 | Time: 1063.4s  ✓ saved
Epoch 04/10 | Train Loss: 2.0633 | Val Loss: 2.4169 | Time: 1041.5s  ✓ saved
Epoch 05/10 | Train Loss: 1.8688 | Val Loss: 2.3482 | Time: 1099.0s  ✓ saved
Epoch 06/10 | Train Loss: 1.7294 | Val Loss: 2.2974 | Time: 438.2s  ✓ saved
Epoch 07/10 | Train Loss: 1.6277 | Val Loss: 2.2910 | Time: 564.1s  ✓ saved
Epoch 08/10 | Train Loss: 1.5396 | Val Loss: 2.2620 | Time: 777.0s  ✓ saved
Epoch 09/10 | Train Loss: 1.4701 | Val Loss: 2.2603 | Time: 617.3s  ✓ saved
Epoch 10/10 | Train Loss: 1.4184 | Val Loss: 2.2573 | Time: 707.3s  ✓ saved


In [6]:
# ── Load best checkpoint & greedy translate ──

model.load_state_dict(torch.load('best_model.pt', map_location=device))
model.eval()


def translate_greedy(sentence, model, src_vocab, tgt_vocab, device, max_len=50):
    """Greedy (argmax) decoding for a single source sentence."""
    model.eval()
    if isinstance(sentence, str):
        sentence   = clean_text(sentence)
        src_indices = src_vocab.encode(sentence)
        src_tensor  = torch.tensor(src_indices, dtype=torch.long,
                                   device=device).unsqueeze(1)
    else:
        src_tensor = sentence.to(device).unsqueeze(1)

    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)

    dec_input  = torch.tensor([tgt_vocab.word2idx[SOS_TOKEN]],
                               dtype=torch.long, device=device)
    eos_idx    = tgt_vocab.word2idx[EOS_TOKEN]
    result_ids = []

    for _ in range(max_len):
        with torch.no_grad():
            pred, hidden, cell = model.decoder(dec_input, hidden, cell)
        top_idx  = pred.argmax(dim=1)
        token_id = top_idx.item()
        if token_id == eos_idx:
            break
        result_ids.append(token_id)
        dec_input = top_idx

    tokens = tgt_vocab.decode(result_ids)
    tokens = [t for t in tokens if t not in (PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN)]
    return ' '.join(tokens)

## 2. BLEU Evaluation (from scratch)
BLEU measures translation quality via clipped n-gram precision weighted with a brevity penalty.
Implemented without any external metric libraries.

In [7]:
def get_ngrams(tokens, n):
    """Return a Counter of all n-grams in `tokens`.

    Parameters
    ----------
    tokens : list[str]
    n      : int

    Returns
    -------
    Counter mapping n-gram tuples to their occurrence count.
    """
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def clipped_precision(hypothesis, reference, n):
    """Compute clipped n-gram precision.

    Each hypothesis n-gram is credited at most as many times as it appears
    in the reference, preventing the model from gaming the score by repeating
    high-probability words.

    Parameters
    ----------
    hypothesis : list[str]   Model output tokens.
    reference  : list[str]   Ground-truth tokens.
    n          : int

    Returns
    -------
    float in [0, 1], or 0.0 if the hypothesis has no n-grams of length n.
    """
    hyp_ngrams = get_ngrams(hypothesis, n)
    ref_ngrams = get_ngrams(reference,  n)

    # Clip each hypothesis n-gram count to the reference count
    clipped_count = sum(min(count, ref_ngrams[ngram])
                        for ngram, count in hyp_ngrams.items())
    total_count = max(len(hypothesis) - n + 1, 0)

    return clipped_count / total_count if total_count > 0 else 0.0


def brevity_penalty(hypothesis, reference):
    """Compute the brevity penalty.

    Returns 1 when hypothesis is at least as long as the reference;
    otherwise returns exp(1 - |ref| / |hyp|) which is < 1.

    Parameters
    ----------
    hypothesis : list[str]
    reference  : list[str]

    Returns
    -------
    float in (0, 1]
    """
    hyp_len = len(hypothesis)
    ref_len = len(reference)

    if hyp_len == 0:
        return 0.0
    if hyp_len >= ref_len:
        return 1.0
    return math.exp(1 - ref_len / hyp_len)


def bleu_score(hypothesis, reference, max_n=4):
    """Compute sentence-level BLEU score.

    Steps:
      1. Compute clipped precision for n = 1 … max_n.
      2. Combine via uniform-weight geometric mean in log space.
      3. Multiply by brevity penalty.

    Parameters
    ----------
    hypothesis : str  Space-separated model output.
    reference  : str  Space-separated ground-truth translation.
    max_n      : int  Highest n-gram order (default 4).

    Returns
    -------
    float in [0, 1]
    """
    hyp_tokens = hypothesis.split()
    ref_tokens = reference.split()

    if len(hyp_tokens) == 0:
        return 0.0

    log_avg = 0.0
    for n in range(1, max_n + 1):
        p = clipped_precision(hyp_tokens, ref_tokens, n)
        if p == 0.0:
            # Any zero precision makes the whole score 0
            return 0.0
        log_avg += math.log(p)

    log_avg /= max_n
    bp = brevity_penalty(hyp_tokens, ref_tokens)
    return bp * math.exp(log_avg)

In [8]:
# ── Sanity check against known example ──
# Expected: ~0.516

hyp_check = "the match was postponed because of the snow"
ref_check = "the match was postponed because it was snowing"

score_check = bleu_score(hyp_check, ref_check)
print(f'Verification BLEU: {score_check:.3f}  (expected ≈ 0.516)')
assert 0.50 < score_check < 0.53, f'Unexpected score: {score_check:.3f}'
print('Sanity check passed ✓')

Verification BLEU: 0.517  (expected ≈ 0.516)
Sanity check passed ✓


In [9]:
# ── Corpus-level BLEU on the validation set ──
# We average sentence-level BLEU scores across all validation pairs.

def evaluate_bleu(model, val_pairs, src_vocab, tgt_vocab, device,
                  translate_fn, max_n=4):
    """Compute average sentence-level BLEU-n scores over val_pairs.

    Parameters
    ----------
    max_n : int  Highest n-gram order to report individually.

    Returns
    -------
    dict mapping 'BLEU-1' … 'BLEU-<max_n>' to float scores.
    """
    scores = defaultdict(float)
    for en_sent, ru_ref in val_pairs:
        prediction = translate_fn(en_sent, model, src_vocab, tgt_vocab, device)
        for n in range(1, max_n + 1):
            scores[n] += bleu_score(prediction, ru_ref, max_n=n)

    n_pairs = len(val_pairs)
    return {f'BLEU-{n}': scores[n] / n_pairs for n in range(1, max_n + 1)}


print('Running BLEU evaluation on validation set …')
bleu_results = evaluate_bleu(model, val_pairs, en_vocab, ru_vocab,
                              device, translate_greedy)

print()
print('=' * 40)
print(f'  BLEU-1 : {bleu_results["BLEU-1"]:.4f}')
print(f'  BLEU-2 : {bleu_results["BLEU-2"]:.4f}')
print(f'  BLEU-4 : {bleu_results["BLEU-4"]:.4f}')
print('=' * 40)

Running BLEU evaluation on validation set …

  BLEU-1 : 0.5177
  BLEU-2 : 0.3508
  BLEU-4 : 0.1280


## 3. Beam Search Decoding
Greedy decoding picks the single best token at each step, which can lead to globally suboptimal translations.
Beam search keeps the top `beam_width` candidate sequences alive at every step, expanding all of them before pruning.

In [10]:
def translate_beam(sentence, model, src_vocab, tgt_vocab, device,
                   beam_width=5, max_len=50):
    """Beam search decoding for a single source sentence.

    Each beam candidate tracks:
      - cumulative log-probability score
      - token sequence generated so far
      - current LSTM hidden and cell states

    Log-probabilities are used throughout to avoid numerical underflow and
    allow direct comparison between candidates.

    Parameters
    ----------
    sentence   : str or torch.Tensor  Raw English string or pre-encoded tensor.
    beam_width : int                  Number of candidates kept at each step.
    max_len    : int                  Maximum decoding steps.

    Returns
    -------
    str  – best completed Russian translation (space-joined tokens).
    """
    model.eval()

    # ── Encode source ──
    if isinstance(sentence, str):
        sentence    = clean_text(sentence)
        src_indices = src_vocab.encode(sentence)
        src_tensor  = torch.tensor(src_indices, dtype=torch.long,
                                   device=device).unsqueeze(1)
    else:
        src_tensor = sentence.to(device).unsqueeze(1)

    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)

    sos_idx = tgt_vocab.word2idx[SOS_TOKEN]
    eos_idx = tgt_vocab.word2idx[EOS_TOKEN]

    # Each beam entry: (neg_log_prob, token_ids, hidden, cell)
    # We negate log-prob so heapq (min-heap) gives us the best candidates.
    beams     = [(0.0, [sos_idx], hidden, cell)]
    completed = []

    for _ in range(max_len):
        if not beams:
            break

        candidates = []
        for neg_score, token_ids, h, c in beams:
            last_token = torch.tensor([token_ids[-1]], dtype=torch.long, device=device)

            with torch.no_grad():
                pred, h_new, c_new = model.decoder(last_token, h, c)

            # Convert logits to log-probabilities
            log_probs = torch.log_softmax(pred, dim=1).squeeze(0)

            # Expand top beam_width tokens
            top_log_probs, top_indices = log_probs.topk(beam_width)

            for log_p, idx in zip(top_log_probs.tolist(), top_indices.tolist()):
                new_score  = neg_score - log_p          # negate for min-heap
                new_tokens = token_ids + [idx]

                if idx == eos_idx:
                    completed.append((new_score, new_tokens))
                else:
                    candidates.append((new_score, new_tokens, h_new, c_new))

        # Keep the best beam_width active candidates
        beams = heapq.nsmallest(beam_width, candidates, key=lambda x: x[0])

        # Early stop if we have enough completed sequences
        if len(completed) >= beam_width:
            break

    # Fall back to active beams if nothing completed
    if not completed:
        completed = [(score, tokens, None, None)
                     for score, tokens, _, _ in beams]

    # Pick best completed sequence (lowest neg log-prob = highest log-prob)
    best_score, best_tokens = min(completed, key=lambda x: x[0])[:2]

    # Strip <sos> and <eos>, then decode
    best_tokens = [t for t in best_tokens
                   if t not in (sos_idx, eos_idx,
                                tgt_vocab.word2idx[PAD_TOKEN],
                                tgt_vocab.word2idx[UNK_TOKEN])]
    tokens = tgt_vocab.decode(best_tokens)
    tokens = [t for t in tokens
              if t not in (PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN)]
    return ' '.join(tokens)

## 4. Comparison Experiment
BLEU-4 and average translation time per sentence for greedy decoding and beam search
with widths 3, 5, and 10, evaluated on 100 random validation sentences.

In [11]:
import matplotlib.pyplot as plt

# Sample 100 validation pairs (fixed seed for reproducibility)
rng = random.Random(SEED)
sample_pairs = rng.sample(val_pairs, 100)

configs = [
    ('Greedy',         lambda s: translate_greedy(s, model, en_vocab, ru_vocab, device)),
    ('Beam  (w=3)',    lambda s: translate_beam(s, model, en_vocab, ru_vocab, device, beam_width=3)),
    ('Beam  (w=5)',    lambda s: translate_beam(s, model, en_vocab, ru_vocab, device, beam_width=5)),
    ('Beam  (w=10)',   lambda s: translate_beam(s, model, en_vocab, ru_vocab, device, beam_width=10)),
]

results = []
for name, translate_fn in configs:
    bleu4_total = 0.0
    times = []
    for en_sent, ru_ref in sample_pairs:
        t0   = time.time()
        pred = translate_fn(en_sent)
        times.append(time.time() - t0)
        bleu4_total += bleu_score(pred, ru_ref, max_n=4)

    avg_bleu4 = bleu4_total / len(sample_pairs)
    avg_time  = sum(times) / len(times)
    results.append((name, avg_bleu4, avg_time))
    print(f'{name:<18} | BLEU-4: {avg_bleu4:.4f} | Avg time: {avg_time*1000:.1f} ms/sent')

# ── Summary table ──
print()
print(f'{'Config':<18} | {'BLEU-4':>8} | {'ms/sent':>10}')
print('-' * 44)
for name, bleu4, t in results:
    print(f'{name:<18} | {bleu4:>8.4f} | {t*1000:>10.1f}')

Greedy             | BLEU-4: 0.1476 | Avg time: 7.8 ms/sent
Beam  (w=3)        | BLEU-4: 0.0881 | Avg time: 19.7 ms/sent
Beam  (w=5)        | BLEU-4: 0.0669 | Avg time: 30.1 ms/sent
Beam  (w=10)       | BLEU-4: 0.0442 | Avg time: 51.1 ms/sent

Config             |   BLEU-4 |    ms/sent
--------------------------------------------
Greedy             |   0.1476 |        7.8
Beam  (w=3)        |   0.0881 |       19.7
Beam  (w=5)        |   0.0669 |       30.1
Beam  (w=10)       |   0.0442 |       51.1


## 5. Translation Examples
Five examples from the validation set comparing greedy decoding, beam search (width=5),
and the reference translation. Examples are chosen to illustrate cases where beam search
helps and — if found — where greedy performs comparably or better.

In [12]:
# ── Score every sample pair and find interesting examples ──
scored_examples = []
for en_sent, ru_ref in sample_pairs:
    greedy_out = translate_greedy(en_sent, model, en_vocab, ru_vocab, device)
    beam_out   = translate_beam(en_sent, model, en_vocab, ru_vocab, device, beam_width=5)

    b_greedy = bleu_score(greedy_out, ru_ref, max_n=4)
    b_beam   = bleu_score(beam_out,   ru_ref, max_n=4)
    delta    = b_beam - b_greedy

    scored_examples.append({
        'en':     en_sent,
        'ref':    ru_ref,
        'greedy': greedy_out,
        'beam':   beam_out,
        'delta':  delta,
        'b_greedy': b_greedy,
        'b_beam':   b_beam,
    })

# Sort by delta: beam beats greedy most → greedy beats beam
scored_examples.sort(key=lambda x: -x['delta'])

# Pick examples: top 3 where beam wins, plus 1 near-tie, plus 1 where greedy wins
beam_wins   = scored_examples[:3]
near_tie    = sorted(scored_examples, key=lambda x: abs(x['delta']))[0:1]
greedy_wins = scored_examples[-1:]

showcase = beam_wins + near_tie + greedy_wins

print('=' * 80)
for i, ex in enumerate(showcase, 1):
    label = ('Beam wins' if ex['delta'] > 0.05
             else 'Greedy wins' if ex['delta'] < -0.05
             else 'Near tie')
    print(f'Example {i}  [{label}]')
    print(f'  EN (source)    : {ex["en"]}')
    print(f'  Reference      : {ex["ref"]}')
    print(f'  Greedy         : {ex["greedy"]}  (BLEU-4: {ex["b_greedy"]:.3f})')
    print(f'  Beam (w=5)     : {ex["beam"]}  (BLEU-4: {ex["b_beam"]:.3f})')
    print('-' * 80)

Example 1  [Near tie]
  EN (source)    : tom and i plan to have three children
  Reference      : мы с томом планируем завести троих детей
  Greedy         : мы с томом планируем три недели  (BLEU-4: 0.430)
  Beam (w=5)     : мы с томом планируем три  (BLEU-4: 0.448)
--------------------------------------------------------------------------------
Example 2  [Near tie]
  EN (source)    : he can sing better than any of us
  Reference      : он поёт лучше чем любой из нас
  Greedy         : он может лучше лучше чем любой  (BLEU-4: 0.000)
  Beam (w=5)     : он поёт лучше лучше чем  (BLEU-4: 0.000)
--------------------------------------------------------------------------------
Example 3  [Near tie]
  EN (source)    : doubleclick the icon
  Reference      : дважды нажмите на значок
  Greedy         : щёлкните  (BLEU-4: 0.000)
  Beam (w=5)     :   (BLEU-4: 0.000)
--------------------------------------------------------------------------------
Example 4  [Near tie]
  EN (source)    : he can s